# SingGAN Implementation
This notebook demonstrates the full pipeline for SingGAN: data preprocessing, model definition, training, and inference.

## 1. Imports and Global Setup
Import required libraries and set global parameters for audio processing and directories.

In [16]:
import os
import glob
import numpy as np
import tensorflow as tf #type:ignore
import librosa #type:ignore
import pyworld as pw #type:ignore
import json
import pickle
import random
import matplotlib.pyplot as plt #type:ignore
from scipy.interpolate import interp1d  #  For precise F0 interpolation
from datetime import datetime  # For TensorBoard logging
import time

# Dataset path
DATASET_PATH = r"C:\Users\npars\Downloads\Compressed\m4singer\m4singer"  # Update to your dataset path

# Audio processing parameters (SingGAN paper, Appendix A, Section 6.1)
SAMPLE_RATE = 24000  # 24kHz as per SingGAN [cite: 319]
N_FFT = 1024  # FFT size for feature extraction
HOP_LENGTH = 128  # Hop length for mel-spectrogram [cite: 324]
WIN_LENGTH = 1024  # Window length, typically matches N_FFT
N_MELS = 80  # Number of mel bands [cite: 324]
F0_MIN = 50  # Minimum F0 value
F0_MAX = 1100  # Maximum F0 value

# pyworld parameters
PW_FFT_SIZE = 1024  # FFT size for pyworld

# Segment length for training
SEGMENT_LENGTH_SECONDS = 2.0  # 2-second segments
GEN_UPSAMPLE_SCALES = [8, 4, 4]  # Upsample scales, product = 128 [cite: 326, 487]
SEGMENT_LENGTH_SAMPLES = int(SAMPLE_RATE * SEGMENT_LENGTH_SECONDS)  # 48000 samples
SEGMENT_LENGTH_FRAMES = SEGMENT_LENGTH_SAMPLES // HOP_LENGTH  # 375 frames

# Directories for preprocessed data
PROCESSED_DATA_DIR = "./processed_data_m4singer"
AUDIO_SEGMENTS_DIR = os.path.join(PROCESSED_DATA_DIR, "audio_segments")
MEL_DIR = os.path.join(PROCESSED_DATA_DIR, "mel")
F0_DIR = os.path.join(PROCESSED_DATA_DIR, "f0")

# Create directories if they don't exist
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(AUDIO_SEGMENTS_DIR, exist_ok=True)
os.makedirs(MEL_DIR, exist_ok=True)
os.makedirs(F0_DIR, exist_ok=True)

## 2. Data Preprocessing
Functions for loading audio, computing mel-spectrograms and F0, and segmenting the dataset.

In [17]:
# --- Preprocessing Functions ---
def load_wav(path, sr):
    """Load WAV file."""
    audio, _ = librosa.load(path, sr=sr)
    return audio

def compute_mel_spectrogram(audio, sr=SAMPLE_RATE):
    """Compute log-mel spectrogram."""
    mel_spec = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH, n_mels=N_MELS
    )
    log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
    return log_mel_spec.astype(np.float32)  # Shape: (n_mels, time)

def compute_f0_pyworld(audio, sr=SAMPLE_RATE, target_length=None):
    """Compute F0 using pyworld."""
    try:
        _f0, t = pw.harvest(audio.astype(np.float64), sr, f0_floor=F0_MIN, f0_ceil=F0_MAX, frame_period=HOP_LENGTH*1000/sr)
        f0 = pw.stonemask(audio.astype(np.float64), _f0, t, sr)  # Refine F0
        if target_length is not None:
            x_old = np.linspace(0, len(f0) - 1, len(f0))
            x_new = np.linspace(0, len(f0) - 1, target_length)
            interp_func = interp1d(x_old, f0, kind='linear', fill_value="extrapolate")
            f0 = interp_func(x_new)
        f0 = np.clip(f0, F0_MIN, F0_MAX).astype(np.float32)
        f0[f0 < F0_MIN] = 0.0  # Mark unvoiced regions
        return f0  # Shape: (time,)
    except Exception as e:
        print(f"F0 computation failed: {e}")
        return np.zeros(target_length if target_length else len(audio) // HOP_LENGTH, dtype=np.float32)

def preprocess_data():
    """Preprocess M4Singer dataset."""
    print("Starting preprocessing...")
    singer_song_dirs = glob.glob(os.path.join(DATASET_PATH, "*#*"))
    print(f"Found {len(singer_song_dirs)} directories.")
    num_dirs_to_process = max(1, int(len(singer_song_dirs) * 0.1))  # Process 10% for testing
    print(f"Processing first {num_dirs_to_process} directories.")
    singer_song_dirs = singer_song_dirs[:num_dirs_to_process]
    data_list = []
    segment_counter = 0

    for dir_path in singer_song_dirs:
        singer_song_name = os.path.basename(dir_path)
        wav_files = glob.glob(os.path.join(dir_path, "*.wav"))
        for wav_path in wav_files:
            base_name = os.path.splitext(os.path.basename(wav_path))[0]
            try:
                audio = load_wav(wav_path, SAMPLE_RATE)
                expected_num_frames = audio.shape[0] // HOP_LENGTH
                log_mel_spec = compute_mel_spectrogram(audio)
                if log_mel_spec.shape[1] != expected_num_frames:
                    log_mel_spec = np.pad(log_mel_spec, ((0, 0), (0, expected_num_frames - log_mel_spec.shape[1])), mode='constant') if log_mel_spec.shape[1] < expected_num_frames else log_mel_spec[:, :expected_num_frames]
                f0 = compute_f0_pyworld(audio, SAMPLE_RATE, target_length=expected_num_frames)
                num_frames = log_mel_spec.shape[1]
                if num_frames < SEGMENT_LENGTH_FRAMES:
                    print(f"Skipping {wav_path}: Too short ({num_frames} frames)")
                    continue
                num_segments = num_frames // SEGMENT_LENGTH_FRAMES
                for i in range(num_segments):
                    start_frame = i * SEGMENT_LENGTH_FRAMES
                    end_frame = start_frame + SEGMENT_LENGTH_FRAMES
                    start_sample = start_frame * HOP_LENGTH
                    end_sample = start_sample + SEGMENT_LENGTH_SAMPLES
                    audio_segment = audio[start_sample:end_sample]
                    segment_id = f"{singer_song_name}_{base_name}_{i:04d}"
                    if len(audio_segment) != SEGMENT_LENGTH_SAMPLES:
                        print(f"Warning: Segment length mismatch for {segment_id}. Expected {SEGMENT_LENGTH_SAMPLES}, got {len(audio_segment)}.")
                        audio_segment = np.pad(audio_segment, (0, SEGMENT_LENGTH_SAMPLES - len(audio_segment))) if len(audio_segment) < SEGMENT_LENGTH_SAMPLES else audio_segment[:SEGMENT_LENGTH_SAMPLES]
                    mel_segment = log_mel_spec[:, start_frame:end_frame]
                    f0_segment = f0[start_frame:end_frame]
                    audio_seg_path = os.path.join(AUDIO_SEGMENTS_DIR, f"{segment_id}.npy")
                    mel_path = os.path.join(MEL_DIR, f"{segment_id}.npy")
                    f0_path = os.path.join(F0_DIR, f"{segment_id}.npy")
                    np.save(audio_seg_path, audio_segment)
                    np.save(mel_path, mel_segment)
                    np.save(f0_path, f0_segment)
                    data_list.append({"segment_id": segment_id, "audio_path": audio_seg_path, "mel_path": mel_path, "f0_path": f0_path})
                    segment_counter += 1
                    if segment_counter % 1000 == 0:
                        print(f"Processed {segment_counter} segments...")
            except Exception as e:
                print(f"Error processing {wav_path}: {e}")
                continue
    data_list_path = os.path.join(PROCESSED_DATA_DIR, "data_list.json")
    with open(data_list_path, 'w') as f:
        json.dump(data_list, f, indent=4)
    print(f"Preprocessing done. Total segments: {segment_counter}")
    print(f"Data list saved to {data_list_path}")

# Run preprocessing if not already done
if not os.path.exists(os.path.join(PROCESSED_DATA_DIR, "data_list.json")):
    print("Starting preprocessing...")
    preprocess_data()
else:
    print("Preprocessed data found. Skipping preprocessing.")

# Load data list for verification
data_list_path = os.path.join(PROCESSED_DATA_DIR, "data_list.json")
with open(data_list_path, 'r') as f:
    data_list = json.load(f)
print("\nChecking preprocessed samples:")
for i in range(min(5, len(data_list))):
    item = data_list[i]
    audio = np.load(item['audio_path'])
    mel = np.load(item['mel_path'])
    f0 = np.load(item['f0_path'])
    print(f"Sample {i}:")
    print(f"  Audio shape: {audio.shape} (Expected: ({SEGMENT_LENGTH_SAMPLES},))")
    print(f"  Mel shape: {mel.shape} (Expected: ({N_MELS}, {SEGMENT_LENGTH_FRAMES}))")
    print(f"  F0 shape: {f0.shape} (Expected: ({SEGMENT_LENGTH_FRAMES},))")
NUM_TOTAL_SEGMENTS = len(data_list)
print(f"Total segments created: {NUM_TOTAL_SEGMENTS}")

Preprocessed data found. Skipping preprocessing.

Checking preprocessed samples:
Sample 0:
  Audio shape: (47872,) (Expected: (48000,))
  Mel shape: (80, 187) (Expected: (80, 375))
  F0 shape: (187,) (Expected: (375,))
Sample 1:
  Audio shape: (47872,) (Expected: (48000,))
  Mel shape: (80, 187) (Expected: (80, 375))
  F0 shape: (187,) (Expected: (375,))
Sample 2:
  Audio shape: (47872,) (Expected: (48000,))
  Mel shape: (80, 187) (Expected: (80, 375))
  F0 shape: (187,) (Expected: (375,))
Sample 3:
  Audio shape: (47872,) (Expected: (48000,))
  Mel shape: (80, 187) (Expected: (80, 375))
  F0 shape: (187,) (Expected: (375,))
Sample 4:
  Audio shape: (47872,) (Expected: (48000,))
  Mel shape: (80, 187) (Expected: (80, 375))
  F0 shape: (187,) (Expected: (375,))
Total segments created: 4755


## 3. Model Definitions
Define the SingGAN generator and discriminator architectures.

In [18]:
# --- Generator and Discriminator ---
from tensorflow.keras.layers import (
    Input, Conv1D, Conv1DTranspose, Dense, LeakyReLU, ReLU,
    Add, Multiply, Concatenate, Reshape, Lambda, UpSampling1D,
    LayerNormalization
)
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K

# Generator hyperparameters
GEN_RESIDUAL_CHANNELS = 64  # [cite: 221, 487]
GEN_KERNEL_SIZE = 5  # [cite: 221, 487]
GEN_DILATIONS = [2**i for i in range(10)]  # Dilations [1, 2, ..., 512] [cite: 220]
GEN_NUM_AFL_BLOCKS = 3  # Number of AFL blocks [cite: 220, 487]
NUM_HARMONICS = 8  # Number of harmonics [cite: 327, 487]

# Discriminator hyperparameters
DISC_RESIDUAL_CHANNELS = 64  # [cite: 493]
DISC_KERNEL_SIZES_SUBBAND = [5, 5, 7, 7]  # Sub-band kernel sizes [cite: 493]
DISC_KERNEL_SIZE_FULLBAND = 3  # Full-band kernel size [cite: 493]
DISC_NUM_LAYERS_SUBBAND = [8, 8, 6, 6]  # Layers per sub-band [cite: 493]
DISC_NUM_LAYERS_FULLBAND = 10  # Full-band layers [cite: 493]
DISC_LEAKY_RELU_SLOPE = 0.2  # LeakyReLU slope [cite: 493]
NUM_DISCRIMINATORS = 5  # 1 global + 4 local [cite: 243, 280]

class SourceExcitationLayer(tf.keras.layers.Layer):
    """Generate sine excitation from F0."""
    def __init__(self, num_harmonics=NUM_HARMONICS, sample_rate=SAMPLE_RATE, **kwargs):
        super().__init__(**kwargs)
        self.num_harmonics = num_harmonics
        self.sample_rate = float(sample_rate)

    def build(self, input_shape):
        self.initial_phases = self.add_weight(
            name='initial_phases', shape=(1, 1, self.num_harmonics),
            initializer=tf.random_uniform_initializer(minval=-np.pi, maxval=np.pi), trainable=False
        )
        super().build(input_shape)

    def call(self, f0_tensor):
        def excitation_fn(f0):
            f0 = tf.cast(f0, tf.float32)  # Shape: [batch_size, time_steps, 1]
            batch_size = tf.shape(f0)[0]
            time_steps = tf.shape(f0)[1]  # 48000 after upsampling
            # Time indices
            time_idx = tf.range(time_steps, dtype=tf.float32) / self.sample_rate  # Shape: [time_steps]
            time_idx = tf.reshape(time_idx, [1, time_steps, 1])  # Shape: [1, time_steps, 1]
            # Harmonic indices
            harmonic_idx = tf.range(1, self.num_harmonics + 1, dtype=tf.float32)
            harmonic_idx = tf.reshape(harmonic_idx, [1, 1, self.num_harmonics])  # Shape: [1, 1, num_harmonics]
            # Compute phase
            f0 = tf.squeeze(f0, axis=-1)  # Shape: [batch_size, time_steps]
            phase_f0 = 2.0 * np.pi * f0[:, :, tf.newaxis] * time_idx  # Shape: [batch_size, time_steps, 1]
            phase_harmonics = harmonic_idx * phase_f0 + self.initial_phases  # Shape: [batch_size, time_steps, num_harmonics]
            # Generate sine waves
            sine_waves = tf.sin(phase_harmonics)  # Shape: [batch_size, time_steps, num_harmonics]
            excitation = tf.reduce_sum(sine_waves, axis=-1)  # Shape: [batch_size, time_steps]
            # Apply voiced/unvoiced mask
            voiced_mask = tf.cast(f0 > 1e-3, tf.float32)  # Shape: [batch_size, time_steps]
            noise = tf.random.normal([batch_size, time_steps], stddev=0.1)  # Shape: [batch_size, time_steps]
            excitation = tf.where(tf.cast(voiced_mask, tf.bool), excitation, noise)  # Shape: [batch_size, time_steps]
            # Add dimension for channel
            return tf.expand_dims(excitation, axis=-1)  # Shape: [batch_size, time_steps, 1]

        # Wrap TensorFlow ops in a Lambda layer
        return tf.keras.layers.Lambda(lambda x: excitation_fn(x))(f0_tensor)

    def get_config(self):
        config = super().get_config()
        config.update({'num_harmonics': self.num_harmonics, 'sample_rate': self.sample_rate})
        return config

class GatedActivationUnit(tf.keras.layers.Layer):
    """Apply gated activation (tanh * sigmoid). [cite: 21]"""
    def __init__(self, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
    
    def call(self, inputs):
        filter_output, gate_output = tf.split(inputs, num_or_size_splits=2, axis=-1)
        return tf.nn.tanh(filter_output) * tf.nn.sigmoid(gate_output)

class AdaptiveFeatureLearningBlock(tf.keras.layers.Layer):
    """AFL block with dilated convolutions and gated activations. [cite: 216]"""
    def __init__(self, residual_channels=GEN_RESIDUAL_CHANNELS, kernel_size=GEN_KERNEL_SIZE, dilations=GEN_DILATIONS, name="AFLBlock", **kwargs):
        super().__init__(name=name, **kwargs)
        self.residual_channels = residual_channels
        self.kernel_size = kernel_size
        self.dilations = dilations
        self.conv_layers = []
        self.gau_layers = []
        self.one_x_one_convs_residual = []
        self.one_x_one_convs_skip = []
        self.initial_conv1x1 = Conv1D(filters=self.residual_channels, kernel_size=1, padding='same', name='initial_conv_1x1')
        self.layer_norm = LayerNormalization(axis=-1)
        for i, dilation in enumerate(self.dilations):
            dilated_conv = Conv1D(filters=self.residual_channels * 2, kernel_size=self.kernel_size, padding='same', dilation_rate=dilation, name=f'dilated_conv_{i}')
            gau = GatedActivationUnit(name=f'gau_{i}')
            conv_residual = Conv1D(filters=self.residual_channels, kernel_size=1, padding='same', name=f'conv_residual_{i}')
            conv_skip = Conv1D(filters=self.residual_channels, kernel_size=1, padding='same', name=f'conv_skip_{i}')
            self.conv_layers.append(dilated_conv)
            self.gau_layers.append(gau)
            self.one_x_one_convs_residual.append(conv_residual)
            self.one_x_one_convs_skip.append(conv_skip)
            
    def call(self, inputs):
        x = self.initial_conv1x1(inputs)
        x = self.layer_norm(x)
        skip_connections = []
        for i in range(len(self.dilations)):
            residual = x
            conv_output = self.conv_layers[i](x)
            gated_output = self.gau_layers[i](conv_output)
            residual_out = self.one_x_one_convs_residual[i](gated_output)
            skip_out = self.one_x_one_convs_skip[i](gated_output)
            x = residual + residual_out
            skip_connections.append(skip_out)
        return tf.add_n(skip_connections)

    def get_config(self):
        config = super().get_config()
        config.update({'residual_channels': self.residual_channels, 'kernel_size': self.kernel_size, 'dilations': self.dilations})
        return config

def build_generator():
    """Build SingGAN generator model. [cite: 200, 201]"""
    mel_input = Input(shape=(N_MELS, SEGMENT_LENGTH_FRAMES), name='mel_input')
    f0_input = Input(shape=(SEGMENT_LENGTH_FRAMES, 1), name='f0_input')
    mel_transposed = tf.keras.layers.Permute((2, 1), name='mel_transpose')(mel_input)
    mel_features = Dense(GEN_RESIDUAL_CHANNELS // 2, name='mel_dense_proj')(mel_transposed)
    mel_features = ReLU(name='mel_relu')(mel_features)
    upsampled_f0 = f0_input
    for i, scale in enumerate(GEN_UPSAMPLE_SCALES):
        upsampled_f0 = UpSampling1D(size=scale, name=f'f0_upsample_{i}')(upsampled_f0)
        upsampled_f0 = Conv1D(filters=1, kernel_size=GEN_KERNEL_SIZE, padding='same', name=f'f0_upsample_conv_{i}')(upsampled_f0)
    excitation_signal = SourceExcitationLayer(num_harmonics=NUM_HARMONICS, sample_rate=SAMPLE_RATE, name='source_excitation')(upsampled_f0)
    # Removed tf.expand_dims here
    mel_upsampled = mel_transposed
    for i, scale in enumerate(GEN_UPSAMPLE_SCALES):
        mel_upsampled = UpSampling1D(size=scale, name=f'mel_upsample_{i}')(mel_upsampled)
    mel_upsampled_proj = Conv1D(filters=GEN_RESIDUAL_CHANNELS - 1, kernel_size=GEN_KERNEL_SIZE, padding='same', name='mel_upsample_proj')(mel_upsampled)
    mel_upsampled_proj = ReLU()(mel_upsampled_proj)
    combined_features = Concatenate(axis=-1, name='combined_features')([excitation_signal, mel_upsampled_proj])
    x = combined_features
    for i in range(GEN_NUM_AFL_BLOCKS):
        afl_block = AdaptiveFeatureLearningBlock(residual_channels=GEN_RESIDUAL_CHANNELS, kernel_size=GEN_KERNEL_SIZE, dilations=GEN_DILATIONS, name=f'AFL_Block_{i+1}')
        x = afl_block(x)
        if i < GEN_NUM_AFL_BLOCKS - 1:
            x = ReLU(name=f'afl_block_relu_{i}')(x)
    waveform = Conv1D(filters=1, kernel_size=GEN_KERNEL_SIZE, padding='same', name='final_conv')(x)
    waveform = tf.keras.layers.Flatten(name='output_flatten')(waveform)
    return Model(inputs=[mel_input, f0_input], outputs=waveform, name='SingGAN_Generator')

class PseudoQuadratureMirrorFilterbank(tf.keras.layers.Layer):
    """Split waveform into sub-bands using PQMF."""
    def __init__(self, num_bands=4, taps=62, cutoff_ratio=0.15, beta=9.0, **kwargs):
        super().__init__(**kwargs)
        self.num_bands = num_bands
        self.taps = taps
        self.cutoff_ratio = cutoff_ratio
        self.beta = beta

    def _get_prototype_filter(self):
        """Design low-pass prototype filter."""
        nyquist_freq = SAMPLE_RATE / 2.0
        cutoff_freq = nyquist_freq / self.num_bands * self.cutoff_ratio
        t = tf.range(-(self.taps // 2), (self.taps + 1) // 2, dtype=tf.float32)
        h = tf.sin(2 * np.pi * cutoff_freq * t / SAMPLE_RATE) / (np.pi * t)
        h = tf.where(tf.equal(t, 0), 2 * cutoff_freq / SAMPLE_RATE, h)
        window = tf.signal.kaiser_window(self.taps, beta=self.beta, dtype=tf.float32)
        h = h * window
        h = h / tf.reduce_sum(tf.abs(h))
        return h

    def build(self, input_shape):
        """Create modulated filters."""
        h = self._get_prototype_filter()
        self.filters = []
        for k in range(self.num_bands):
            modulation = tf.cos((2 * k + 1) * np.pi / (2 * self.num_bands) * (tf.range(float(self.taps)) + 0.5))
            filter_k = h * modulation
            filter_k = tf.reshape(filter_k, [self.taps, 1, 1])
            self.filters.append(filter_k)
        super().build(input_shape)

    def call(self, inputs):
        """Apply PQMF to input waveform and add channel dimension."""
        inputs = tf.expand_dims(inputs, axis=-1)  # Shape: [batch_size, time_steps, 1]
        sub_bands = []
        for k in range(self.num_bands):
            padded_inputs = tf.pad(inputs, [[0, 0], [self.taps // 2, self.taps // 2], [0, 0]], mode='REFLECT')
            filtered = tf.nn.conv1d(padded_inputs, self.filters[k], stride=1, padding='VALID')
            sub_band = filtered[:, ::self.num_bands, 0]  # Shape: [batch_size, time_steps // num_bands]
            sub_band = tf.expand_dims(sub_band, axis=-1)  # Shape: [batch_size, time_steps // num_bands, 1]
            sub_bands.append(sub_band)
        return sub_bands

    def get_config(self):
        config = super().get_config()
        config.update({
            'num_bands': self.num_bands,
            'taps': self.taps,
            'cutoff_ratio': self.cutoff_ratio,
            'beta': self.beta
        })
        return config

class DiscriminatorResidualBlock(tf.keras.layers.Layer):
    """Residual block for discriminator with Conv1D and LeakyReLU."""
    def __init__(self, channels, kernel_size, dilation, name=None, **kwargs):
        super().__init__(name=name, **kwargs)
        self.conv1 = Conv1D(channels, kernel_size, padding='same', dilation_rate=dilation)
        self.lrelu1 = LeakyReLU(DISC_LEAKY_RELU_SLOPE)
        self.conv2 = Conv1D(channels, kernel_size, padding='same', dilation_rate=1)
        self.lrelu2 = LeakyReLU(DISC_LEAKY_RELU_SLOPE)
        self.res_conv = Conv1D(channels, kernel_size=1, padding='same')

    def call(self, inputs):
        x = self.conv1(inputs)
        x = self.lrelu1(x)
        x = self.conv2(x)
        x = self.lrelu2(x)
        if inputs.shape[-1] != x.shape[-1]:
            inputs = self.res_conv(inputs)
        return inputs + x

def build_discriminator():
    """Build discriminator with global and local branches."""
    waveform_input = Input(shape=(SEGMENT_LENGTH_SAMPLES,), name='waveform_input')
    # Wrap tf.expand_dims in a Lambda layer
    x_global = tf.keras.layers.Lambda(lambda x: tf.expand_dims(x, axis=-1))(waveform_input)
    x_global = Conv1D(DISC_RESIDUAL_CHANNELS, DISC_KERNEL_SIZE_FULLBAND, padding='same', name='global_initial_conv')(x_global)
    x_global = LeakyReLU(DISC_LEAKY_RELU_SLOPE)(x_global)
    global_features = []
    for i in range(DISC_NUM_LAYERS_FULLBAND):
        x_global = Conv1D(DISC_RESIDUAL_CHANNELS, DISC_KERNEL_SIZE_FULLBAND, padding='same', name=f'global_conv_{i}')(x_global)
        x_global = LeakyReLU(DISC_LEAKY_RELU_SLOPE)(x_global)
        global_features.append(x_global)
    global_output = Conv1D(1, 1, padding='same', name='global_final_conv')(x_global)
    global_validity = tf.keras.layers.GlobalAveragePooling1D(name='global_avg_pool')(global_output)
    pqmf = PseudoQuadratureMirrorFilterbank(num_bands=4, taps=62, cutoff_ratio=0.15, beta=9.0, name='pqmf')
    sub_bands = pqmf(waveform_input)
    discriminator_outputs = [global_validity]
    all_discriminator_features = [global_features]
    for i, sub_band in enumerate(sub_bands):
        x_local = sub_band  # Already has shape [batch_size, time_steps // num_bands, 1]
        kernel_size_local = DISC_KERNEL_SIZES_SUBBAND[i]
        num_layers_local = DISC_NUM_LAYERS_SUBBAND[i]
        x_local = Conv1D(DISC_RESIDUAL_CHANNELS, kernel_size_local, padding='same', name=f'local_{i+1}_initial_conv')(x_local)
        x_local = LeakyReLU(DISC_LEAKY_RELU_SLOPE)(x_local)
        local_features = []
        dilations_for_local = [2**j for j in range(num_layers_local)]
        for j in range(num_layers_local):
            dilation_rate = dilations_for_local[j % len(dilations_for_local)]
            x_local = Conv1D(DISC_RESIDUAL_CHANNELS, kernel_size_local, padding='same', dilation_rate=dilation_rate, name=f'local_{i+1}_conv_{j}')(x_local)
            x_local = LeakyReLU(DISC_LEAKY_RELU_SLOPE)(x_local)
            local_features.append(x_local)
        local_output = Conv1D(1, 1, padding='same', name=f'local_{i+1}_final_conv')(x_local)
        local_validity = tf.keras.layers.GlobalAveragePooling1D(name=f'local_{i+1}_avg_pool')(local_output)
        discriminator_outputs.append(local_validity)
        all_discriminator_features.append(local_features)
    return Model(inputs=waveform_input, outputs=[discriminator_outputs, all_discriminator_features], name='SingGAN_Discriminator')

## 4. Loss Functions
Define multi-resolution STFT, Mel, auxiliary, and feature matching losses for training.

In [19]:
# --- Loss Functions ---
LAMBDA_AUX = 0.5  # Weight for auxiliary loss [cite: 268]
LAMBDA_ADV_GEN = 4.0  # Weight for generator adversarial loss [cite: 279]
LAMBDA_FM = 10.0  # Weight for feature matching loss [cite: 279]
STFT_PARAMS = [(512, 50, 240), (1024, 120, 600), (2048, 240, 1200)]
MEL_PARAMS = [(2048, 270, 1080), (4096, 540, 2160)]

def stft_loss(y_true, y_pred, n_fft, hop_length, win_length):
    """Compute STFT loss (spectral convergence + log magnitude). [cite: 250, 254]"""
    y_true, y_pred = tf.cast(y_true, tf.float32), tf.cast(y_pred, tf.float32)
    stft_true = tf.signal.stft(y_true, frame_length=win_length, frame_step=hop_length, fft_length=n_fft, window_fn=tf.signal.hann_window)
    stft_pred = tf.signal.stft(y_pred, frame_length=win_length, frame_step=hop_length, fft_length=n_fft, window_fn=tf.signal.hann_window)
    mag_true, mag_pred = tf.abs(stft_true), tf.abs(stft_pred)
    diff_sc = tf.norm(mag_true - mag_pred, ord='fro', axis=(-2, -1)) / (tf.norm(mag_true, ord='fro', axis=(-2, -1)) + 1e-8)
    log_mag_loss = tf.reduce_mean(tf.abs(tf.math.log(mag_true + 1e-8) - tf.math.log(mag_pred + 1e-8)), axis=(-2, -1))
    return tf.reduce_mean(diff_sc), tf.reduce_mean(log_mag_loss)

def multi_resolution_stft_loss(y_true, y_pred):
    """Compute multi-resolution STFT loss. [cite: 256]"""
    total_sc_loss, total_log_mag_loss = 0.0, 0.0
    for n_fft, hop_length, win_length in STFT_PARAMS:
        sc_loss, log_mag_loss = stft_loss(y_true, y_pred, n_fft, hop_length, win_length)
        total_sc_loss += sc_loss
        total_log_mag_loss += log_mag_loss
    return (total_sc_loss / len(STFT_PARAMS)) + (total_log_mag_loss / len(STFT_PARAMS))

def mel_loss(y_true, y_pred, n_fft, hop_length, win_length, n_mels=N_MELS, sample_rate=SAMPLE_RATE):
    """Compute Mel-spectrogram loss. [cite: 259, 261]"""
    y_true, y_pred = tf.cast(y_true, tf.float32), tf.cast(y_pred, tf.float32)
    stft_true = tf.signal.stft(y_true, frame_length=win_length, frame_step=hop_length, fft_length=n_fft, window_fn=tf.signal.hann_window)
    stft_pred = tf.signal.stft(y_pred, frame_length=win_length, frame_step=hop_length, fft_length=n_fft, window_fn=tf.signal.hann_window)
    mag_true, mag_pred = tf.abs(stft_true), tf.abs(stft_pred)
    num_spectrogram_bins = stft_true.shape[-1]
    linear_to_mel_weight_matrix = tf.signal.linear_to_mel_weight_matrix(
        num_mel_bins=n_mels, num_spectrogram_bins=num_spectrogram_bins, sample_rate=sample_rate,
        lower_edge_hertz=0.0, upper_edge_hertz=sample_rate / 2.0
    )
    mel_spec_true = tf.tensordot(mag_true, linear_to_mel_weight_matrix, 1)
    mel_spec_pred = tf.tensordot(mag_pred, linear_to_mel_weight_matrix, 1)
    diff_sc = tf.norm(mel_spec_true - mel_spec_pred, ord='fro', axis=(-2, -1)) / (tf.norm(mel_spec_true, ord='fro', axis=(-2, -1)) + 1e-8)
    log_mel_true = tf.math.log(mel_spec_true + 1e-8)
    log_mel_pred = tf.math.log(mel_spec_pred + 1e-8)
    log_mag_loss = tf.reduce_mean(tf.abs(log_mel_true - log_mel_pred), axis=(-2, -1))
    return tf.reduce_mean(diff_sc), tf.reduce_mean(log_mag_loss)

def multi_resolution_mel_loss(y_true, y_pred):
    """Compute multi-resolution Mel-spectrogram loss. [cite: 264]"""
    total_sc_loss, total_log_mag_loss = 0.0, 0.0
    for n_fft, hop_length, win_length in MEL_PARAMS:
        sc_loss, log_mag_loss = mel_loss(y_true, y_pred, n_fft, hop_length, win_length)
        total_sc_loss += sc_loss
        total_log_mag_loss += log_mag_loss
    return (total_sc_loss / len(MEL_PARAMS)) + (total_log_mag_loss / len(MEL_PARAMS))

def auxiliary_loss(y_true, y_pred):
    """Combine Mel and STFT losses. [cite: 266]"""
    stft_loss_val = multi_resolution_stft_loss(y_true, y_pred)
    mel_loss_val = multi_resolution_mel_loss(y_true, y_pred)
    return mel_loss_val + LAMBDA_AUX * stft_loss_val

def feature_matching_loss(discriminator_real_features_list, discriminator_fake_features_list):
    """Compute feature matching loss for local discriminators. [cite: 272]"""
    total_fm_loss = 0.0
    for i in range(1, NUM_DISCRIMINATORS):
        real_features_k = discriminator_real_features_list[i]
        fake_features_k = discriminator_fake_features_list[i]
        for real_feat, fake_feat in zip(real_features_k, fake_features_k):
            total_fm_loss += tf.reduce_mean(tf.abs(real_feat - fake_feat))
    return total_fm_loss

## 5. Data Loading
Load preprocessed segments and create TensorFlow datasets for training.

In [20]:
# --- Data Loading ---
BATCH_SIZE = 8  # Batch size for training [cite: 339]
SHUFFLE_BUFFER_SIZE = 1000  # Buffer size for shuffling

def load_segment_tf(segment_info):
    """Load segment data from disk."""
    mel_path = segment_info['mel_path']
    f0_path = segment_info['f0_path']
    audio_path = segment_info['audio_path']
    def _load_numpy_data(mel_p, f0_p, audio_p):
        mel_np = np.load(mel_p.numpy()).astype(np.float32)
        f0_np = np.load(f0_p.numpy()).astype(np.float32)
        audio_np = np.load(audio_p.numpy()).astype(np.float32)
        if mel_np.shape[1] < SEGMENT_LENGTH_FRAMES:
            mel_np = np.pad(mel_np, ((0, 0), (0, SEGMENT_LENGTH_FRAMES - mel_np.shape[1])), mode='constant')
        elif mel_np.shape[1] > SEGMENT_LENGTH_FRAMES:
            mel_np = mel_np[:, :SEGMENT_LENGTH_FRAMES]
        if f0_np.shape[0] < SEGMENT_LENGTH_FRAMES:
            f0_np = np.pad(f0_np, (0, SEGMENT_LENGTH_FRAMES - f0_np.shape[0]), mode='constant')
        elif f0_np.shape[0] > SEGMENT_LENGTH_FRAMES:
            f0_np = f0_np[:SEGMENT_LENGTH_FRAMES]
        f0_np = np.reshape(f0_np, (SEGMENT_LENGTH_FRAMES, 1))
        if audio_np.shape[0] < SEGMENT_LENGTH_SAMPLES:
            audio_np = np.pad(audio_np, (0, SEGMENT_LENGTH_SAMPLES - audio_np.shape[0]), mode='constant')
        elif audio_np.shape[0] > SEGMENT_LENGTH_SAMPLES:
            audio_np = audio_np[:SEGMENT_LENGTH_SAMPLES]
        return mel_np, f0_np, audio_np
    mel, f0, audio = tf.py_function(_load_numpy_data, [mel_path, f0_path, audio_path], [tf.float32, tf.float32, tf.float32])
    mel.set_shape([N_MELS, SEGMENT_LENGTH_FRAMES])
    f0.set_shape([SEGMENT_LENGTH_FRAMES, 1])
    audio.set_shape([SEGMENT_LENGTH_SAMPLES])
    return (mel, f0), audio

def create_dataset():
    """Create tf.data.Dataset for training."""
    data_list_path = os.path.join(PROCESSED_DATA_DIR, "data_list.json")
    with open(data_list_path, 'r') as f:
        data_list_info = json.load(f)
    dataset = tf.data.Dataset.from_generator(
        lambda: data_list_info,
        output_signature={
            "segment_id": tf.TensorSpec(shape=(), dtype=tf.string),
            "audio_path": tf.TensorSpec(shape=(), dtype=tf.string),
            "mel_path": tf.TensorSpec(shape=(), dtype=tf.string),
            "f0_path": tf.TensorSpec(shape=(), dtype=tf.string),
        }
    )
    dataset = dataset.map(load_segment_tf, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.shuffle(SHUFFLE_BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
    return dataset

# Create dataset
print("\nCreating dataset...")
train_dataset = create_dataset()
for (mel_batch, f0_batch), audio_batch in train_dataset.take(1):
    print(f"Mel batch shape: {mel_batch.shape}")
    print(f"F0 batch shape: {f0_batch.shape}")
    print(f"Audio batch shape: {audio_batch.shape}")
print("Dataset check complete.")


Creating dataset...
Mel batch shape: (8, 80, 375)
F0 batch shape: (8, 375, 1)
Audio batch shape: (8, 48000)
Dataset check complete.


## 6. Training Setup
Configure optimizers, checkpoints, metrics, and the training loop.

In [21]:
# Instantiate models
generator = build_generator()
discriminator = build_discriminator()

generator.summary()
discriminator.summary()

Model: "SingGAN_Generator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ f0_input            │ (None, 375, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ f0_upsample_0       │ (None, 3000, 1)   │          0 │ f0_input[0][0]    │
│ (UpSampling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_input           │ (None, 80, 375)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ f0_upsample_conv_0  │ (None, 3000, 1)   │          6 │ f0_upsample_0[0]… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_transpose       │ (None, 375, 80)   │          0 │ mel_input[0][0]   │
│ (Permute)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ f0_upsample_1       │ (None, 12000, 1)  │          0 │ f0_upsample_conv… │
│ (UpSampling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_upsample_0      │ (None, 3000, 80)  │          0 │ mel_transpose[0]… │
│ (UpSampling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ f0_upsample_conv_1  │ (None, 12000, 1)  │          6 │ f0_upsample_1[0]… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_upsample_1      │ (None, 12000, 80) │          0 │ mel_upsample_0[0… │
│ (UpSampling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ f0_upsample_2       │ (None, 48000, 1)  │          0 │ f0_upsample_conv… │
│ (UpSampling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_upsample_2      │ (None, 48000, 80) │          0 │ mel_upsample_1[0… │
│ (UpSampling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ f0_upsample_conv_2  │ (None, 48000, 1)  │          6 │ f0_upsample_2[0]… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mel_upsample_proj   │ (None, 48000, 63) │     25,263 │ mel_upsample_2[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ source_excitation   │ (None, 48000, 1)  │          8 │ f0_upsample_conv… │
│ (SourceExcitationL… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 48000, 63) │          0 │ mel_upsample_pro… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined_features   │ (None, 48000, 64) │          0 │ source_excitatio… │
│ (Concatenate)       │                   │            │ re_lu_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ AFL_Block_1         │ (None, 48000, 64) │    498,368 │ combined_feature… │
│ (AdaptiveFeatureLe… │                   │            │                 

 Total params: 1,520,714 (5.80 MB)

 Trainable params: 1,520,706 (5.80 MB)

 Non-trainable params: 8 (32.00 B)

Model: "SingGAN_Discriminator"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ waveform_input      │ (None, 48000)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_7 (Lambda)   │ (None, 48000, 1)  │          0 │ waveform_input[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_initial_conv │ (None, 48000, 64) │        256 │ lambda_7[0][0]    │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_43      │ (None, 48000, 64) │          0 │ global_initial_c… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_conv_0       │ (None, 48000, 64) │     12,352 │ leaky_re_lu_43[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_44      │ (None, 48000, 64) │          0 │ global_conv_0[0]… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pqmf                │ [(None, 12001,    │          0 │ waveform_input[0… │
│ (PseudoQuadratureM… │ 1), (None, 12001, │            │                   │
│                     │ 1), (None, 12001, │            │                   │
│                     │ 1), (None, 12001, │            │                   │
│                     │ 1)]               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_conv_1       │ (None, 48000, 64) │     12,352 │ leaky_re_lu_44[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_1_initial_co… │ (None, 12001, 64) │        384 │ pqmf[0][0]        │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_2_initial_co… │ (None, 12001, 64) │        384 │ pqmf[0][1]        │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_45      │ (None, 48000, 64) │          0 │ global_conv_1[0]… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_54      │ (None, 12001, 64) │          0 │ local_1_initial_… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_63      │ (None, 12001, 64) │          0 │ local_2_initial_… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_conv_2       │ (None, 48000, 64) │     12,352 │ leaky_re_lu_45[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_1_conv_0      │ (None, 12001, 64) │     20,544 │ leaky_re_lu_54[0… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_2_conv_0      │ (None, 12001, 64) │     20,544 │ leaky_re_lu_63[0… │
│ (Conv1D)            │                   │            │                 

 Total params: 799,429 (3.05 MB)

 Trainable params: 799,429 (3.05 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# --- Training Setup ---
EPOCHS = 2  # Training epochs
LEARNING_RATE_G = 1e-4  # Generator learning rate
LEARNING_RATE_D = 1e-4  # Discriminator learning rate
BATCHES_PER_EPOCH = NUM_TOTAL_SEGMENTS // BATCH_SIZE if NUM_TOTAL_SEGMENTS > 0 else 100
CHECKPOINT_DIR = "./checkpoints_singgan"
LOG_DIR = os.path.join("./logs_singgan", datetime.now().strftime("%Y%m%d-%H%M%S"))
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# --- Early Stopping and Learning Rate Scheduling ---
class EarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, monitor='train_loss_gen', patience=20, min_delta=0.001):
        super(EarlyStopping, self).__init__()
        self.monitor = monitor
        self.patience = patience
        self.min_delta = min_delta
        self.best = None
        self.wait = 0
        self.stopped_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        current = logs.get(self.monitor)
        if self.best is None:
            self.best = current
        elif current > self.best - self.min_delta:
            self.wait += 1
            if self.wait >= self.patience:
                self.stopped_epoch = epoch
                self.model.stop_training = True
        else:
            self.best = current
            self.wait = 0

lr_scheduler = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=LEARNING_RATE_G,
    decay_steps=BATCHES_PER_EPOCH * 10,  # Decay every 10 epochs
    decay_rate=0.9,
    staircase=True
)
generator_optimizer = tf.keras.optimizers.Adam(learning_rate=lr_scheduler, beta_1=0.9, beta_2=0.98, epsilon=1e-9)
discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE_D, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

# به‌روزرسانی چک‌پوینت بدون اضافه کردن scheduler
checkpoint = tf.train.Checkpoint(step=tf.Variable(0), generator_optimizer=generator_optimizer, discriminator_optimizer=discriminator_optimizer, generator=generator, discriminator=discriminator)
checkpoint_manager = tf.train.CheckpointManager(checkpoint, CHECKPOINT_DIR, max_to_keep=3)

train_loss_gen = tf.keras.metrics.Mean(name='train_loss_gen')
train_loss_disc = tf.keras.metrics.Mean(name='train_loss_disc')
train_gen_adv_loss = tf.keras.metrics.Mean(name='train_gen_adv_loss')
train_disc_real_loss = tf.keras.metrics.Mean(name='train_disc_real_loss')
train_disc_fake_loss = tf.keras.metrics.Mean(name='train_disc_fake_loss')
train_aux_loss = tf.keras.metrics.Mean(name='train_aux_loss')
train_fm_loss = tf.keras.metrics.Mean(name='train_fm_loss')

summary_writer = tf.summary.create_file_writer(LOG_DIR)
WARMUP_STEPS = BATCHES_PER_EPOCH * 5

@tf.function
def train_step(mel, f0, real_audio, global_step):
    """Perform one training step."""
    mel, f0, real_audio = tf.cast(mel, tf.float32), tf.cast(f0, tf.float32), tf.cast(real_audio, tf.float32)
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_audio = generator([mel, f0], training=True)
        if generated_audio.shape[1] != SEGMENT_LENGTH_SAMPLES:
            generated_audio = generated_audio[:, :SEGMENT_LENGTH_SAMPLES] if generated_audio.shape[1] > SEGMENT_LENGTH_SAMPLES else tf.pad(generated_audio, [[0, 0], [0, SEGMENT_LENGTH_SAMPLES - generated_audio.shape[1]]])
        disc_real_outputs, disc_real_features = discriminator(real_audio, training=True)
        disc_fake_outputs, disc_fake_features = discriminator(generated_audio, training=True)
        disc_loss_total = 0.0
        disc_real_losses, disc_fake_losses = [], []
        for k in range(NUM_DISCRIMINATORS):
            disc_loss_real_k = tf.reduce_mean(tf.square(disc_real_outputs[k] - 1))
            disc_loss_fake_k = tf.reduce_mean(tf.square(disc_fake_outputs[k]))
            disc_loss_total += (disc_loss_real_k + disc_loss_fake_k)
            disc_real_losses.append(disc_loss_real_k)
            disc_fake_losses.append(disc_loss_fake_k)
        disc_loss = disc_loss_total
        gen_loss_adv_total = 0.0
        for k in range(NUM_DISCRIMINATORS):
            gen_loss_adv_total += tf.reduce_mean(tf.square(disc_fake_outputs[k] - 1))
        gen_loss_adv = gen_loss_adv_total / NUM_DISCRIMINATORS
        gen_loss_aux = auxiliary_loss(real_audio, generated_audio)
        gen_loss_fm = feature_matching_loss(disc_real_features, disc_fake_features)
        gen_loss = gen_loss_aux + LAMBDA_ADV_GEN * gen_loss_adv + LAMBDA_FM * gen_loss_fm
    if global_step >= WARMUP_STEPS:
        disc_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
        discriminator_optimizer.apply_gradients(zip(disc_gradients, discriminator.trainable_variables))
    gen_gradients = gen_tape.gradient(gen_loss, generator.trainable_variables)
    generator_optimizer.apply_gradients(zip(gen_gradients, generator.trainable_variables))
    train_loss_gen.update_state(gen_loss)
    train_loss_disc.update_state(disc_loss)
    train_gen_adv_loss.update_state(gen_loss_adv)
    train_aux_loss.update_state(gen_loss_aux)
    train_fm_loss.update_state(gen_loss_fm)
    train_disc_real_loss.update_state(tf.reduce_mean(disc_real_losses))
    train_disc_fake_loss.update_state(tf.reduce_mean(disc_fake_losses))
    return gen_loss, disc_loss

def train(dataset, epochs):
    """Main training loop."""
    if checkpoint_manager.latest_checkpoint:
        checkpoint.restore(checkpoint_manager.latest_checkpoint)
        print(f"Restored from checkpoint at step {checkpoint.step.numpy()}.")
    else:
        print("No checkpoint found. Starting training from scratch.")
    initial_step = checkpoint.step.numpy()
    early_stopping = EarlyStopping(monitor='train_loss_gen', patience=20, min_delta=0.001)
    for epoch in range(epochs):
        start_time = time.time()
        train_loss_gen.reset_state()
        train_loss_disc.reset_state()
        train_gen_adv_loss.reset_state()
        train_aux_loss.reset_state()
        train_fm_loss.reset_state()
        train_disc_real_loss.reset_state()
        train_disc_fake_loss.reset_state()
        print(f"\nEpoch {epoch + 1}/{epochs}")
        print("-" * 30)
        batch_count = 0
        for (mel_batch, f0_batch), audio_batch in dataset:
            current_step = initial_step + (epoch * BATCHES_PER_EPOCH) + batch_count
            checkpoint.step.assign(current_step)
            gen_loss, disc_loss = train_step(mel_batch, f0_batch, audio_batch, current_step)
            batch_count += 1
            if batch_count % 10 == 0:
                print(f'  Step: {current_step}, Batch: {batch_count}/{BATCHES_PER_EPOCH}, '
                      f'Gen Loss: {train_loss_gen.result():.4f}, Disc Loss: {train_loss_disc.result():.4f}, '
                      f'Aux Loss: {train_aux_loss.result():.4f}, FM Loss: {train_fm_loss.result():.4f}')
            if batch_count >= BATCHES_PER_EPOCH:
                break
        end_time = time.time()
        print(f'Epoch {epoch + 1} completed. Time: {end_time - start_time:.2f} sec')
        ckpt_save_path = checkpoint_manager.save()
        print(f'Saving checkpoint for epoch {epoch + 1} at {ckpt_save_path}')
        with summary_writer.as_default():
            tf.summary.scalar('Generator/Total_Loss', train_loss_gen.result(), step=checkpoint.step.numpy())
            tf.summary.scalar('Generator/Adversarial_Loss', train_gen_adv_loss.result(), step=checkpoint.step.numpy())
            tf.summary.scalar('Generator/Auxiliary_Loss', train_aux_loss.result(), step=checkpoint.step.numpy())
            tf.summary.scalar('Generator/Feature_Matching_Loss', train_fm_loss.result(), step=checkpoint.step.numpy())
            tf.summary.scalar('Discriminator/Total_Loss', train_loss_disc.result(), step=checkpoint.step.numpy())
            tf.summary.scalar('Discriminator/Real_Loss', train_disc_real_loss.result(), step=checkpoint.step.numpy())
            tf.summary.scalar('Discriminator/Fake_Loss', train_disc_fake_loss.result(), step=checkpoint.step.numpy())
            summary_writer.flush()
        early_stopping.on_epoch_end(epoch, {'train_loss_gen': train_loss_gen.result()})
        if early_stopping.stopped_epoch > 0:
            print(f"Early stopping triggered at epoch {early_stopping.stopped_epoch + 1}")
            break
        print(f'Summary: Gen Loss: {train_loss_gen.result():.4f}, Disc Loss: {train_loss_disc.result():.4f}')
# Start training
print("Starting Training...")
train(train_dataset, EPOCHS)
print("Training finished.")

Starting Training...
No checkpoint found. Starting training from scratch.

Epoch 1/2
------------------------------
  Step: 9, Batch: 10/594, Gen Loss: 38.5688, Disc Loss: 4.9991, Aux Loss: 34.1114, FM Loss: 0.0467
  Step: 19, Batch: 20/594, Gen Loss: 28.9488, Disc Loss: 4.9991, Aux Loss: 24.6185, FM Loss: 0.0337
  Step: 29, Batch: 30/594, Gen Loss: 23.2400, Disc Loss: 4.9991, Aux Loss: 18.9945, FM Loss: 0.0250
  Step: 39, Batch: 40/594, Gen Loss: 21.5803, Disc Loss: 4.9991, Aux Loss: 17.3878, FM Loss: 0.0197
  Step: 49, Batch: 50/594, Gen Loss: 19.6453, Disc Loss: 4.9991, Aux Loss: 15.4851, FM Loss: 0.0164
  Step: 59, Batch: 60/594, Gen Loss: 18.0006, Disc Loss: 4.9991, Aux Loss: 13.8623, FM Loss: 0.0142
  Step: 69, Batch: 70/594, Gen Loss: 17.3801, Disc Loss: 4.9991, Aux Loss: 13.2580, FM Loss: 0.0126
  Step: 79, Batch: 80/594, Gen Loss: 18.1571, Disc Loss: 4.9991, Aux Loss: 14.0480, FM Loss: 0.0113
  Step: 89, Batch: 90/594, Gen Loss: 17.6230, Disc Loss: 4.9991, Aux Loss: 13.5232, F

## 7. Inference and Audio Output
Generate audio from the trained model and save output samples for listening.

In [ ]:
# --- Audio Output Check ---
import soundfile as sf  # For saving audio

print("\n--- Checking Audio Output ---")
data_list_path = os.path.join(PROCESSED_DATA_DIR, "data_list.json")
with open(data_list_path, 'r') as f:
    data_list_inference = json.load(f)
sample_index = 0
segment_info_inference = data_list_inference[sample_index]
(mel_inf, f0_inf), real_audio_inf = load_segment_tf(segment_info_inference)
print(f"Loaded shapes for inference:")
print(f"  Mel: {mel_inf.shape}")
print(f"  F0: {f0_inf.shape}")
print(f"  Real Audio: {real_audio_inf.shape}")
mel_batch_inf = tf.expand_dims(mel_inf, axis=0)
f0_batch_inf = tf.expand_dims(f0_inf, axis=0)
generated_audio_inf = generator([mel_batch_inf, f0_batch_inf], training=False)
generated_audio_np = generated_audio_inf.numpy()[0]
if generated_audio_np.shape[0] != SEGMENT_LENGTH_SAMPLES:
    print(f"Warning: Generated audio length mismatch. Expected {SEGMENT_LENGTH_SAMPLES}, got {generated_audio_np.shape[0]}.")
    generated_audio_np = generated_audio_np[:SEGMENT_LENGTH_SAMPLES]
print(f"Shape of generated audio: {generated_audio_np.shape}")
real_audio_path_save = f"real_sample_{sample_index}.wav"
sf.write(real_audio_path_save, real_audio_inf.numpy(), SAMPLE_RATE)
generated_audio_path_save = f"generated_sample_{sample_index}.wav"
sf.write(generated_audio_path_save, generated_audio_np, SAMPLE_RATE)
print(f"Audio saved: {real_audio_path_save}, {generated_audio_path_save}")
print("Audio output check complete.")